# Week 10: Poker RLM — Environment, Heuristic & Training Pipeline

**Progress since Week 8:**
- Pivoted to poker as the application domain (long context genuinely matters)
- Built poker environment with hand evaluation and game state
- Heuristic bot with 3-step retrieve → compute → decide reasoning
- Opponent modeling with 5 archetypes (rock/TAG/LAG/fish/maniac)
- Training pipeline adapted for poker (BC + REINFORCE)

This notebook demonstrates the full poker RLM pipeline.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from src.poker.environment import Card, Deck, HandEvaluator, GameState, OPPONENT_ARCHETYPES
from src.poker.heuristic import HeuristicPokerBot, preflop_tier, _hand_key
from src.poker.tasks import generate_poker_task, generate_poker_task_with_trace
from src.poker.agents import PokerHeuristicAgent, PokerLLMAgent
from src.poker.rewards import compute_poker_reward, compute_poker_reward_simple, parse_action
from src.poker.evaluation import PokerEvaluationFramework
from src.poker.training import (
    collect_poker_trajectories,
    collect_poker_trajectories_with_traces,
    PokerBCTrainer,
    PokerReinforceTrainer,
)

import random
random.seed(42)

print("All imports successful.")

# Check for HF_TOKEN and GPU
HF_TOKEN_SET = bool(os.environ.get('HF_TOKEN'))
print(f"HF_TOKEN: {'set' if HF_TOKEN_SET else 'not set'}")

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    print(f"GPU: {'available' if GPU_AVAILABLE else 'not available'}")
except ImportError:
    GPU_AVAILABLE = False
    print("PyTorch not installed — training sections will be skipped")

## 2. Problem Formulation

### Why Poker?

Poker is an ideal domain for the RLM framework because:
- **Long context matters**: Optimal play requires parsing multi-hand histories to identify opponent tendencies
- **Computation helps**: Pot odds, equity estimation, and opponent stats require actual calculation
- **The retrieve → compute → decide flow is natural**: Read history → analyze hand → make decision

### Objective

**Phase 1 (Behavior Cloning):** Train LLM to replicate the heuristic bot's 3-step reasoning:
$$\min_\theta \; -\sum_i \log \pi_\theta(\text{code}_i \mid \text{state}_i)$$

**Phase 2 (REINFORCE):** Optimize for EV-based reward:
$$R = 0.5 \cdot \text{action\_match} + 0.3 \cdot \text{EV} + 0.2 \cdot \text{reasoning\_bonus} - \text{step\_penalty}$$

## 3. Opponent Archetypes

In [ ]:
print(f"{'Archetype':<10} {'VPIP':>6} {'PFR':>6} {'Agg':>6} {'Fold-CB':>8} {'3-Bet':>6}")
print("-" * 48)
for name, profile in OPPONENT_ARCHETYPES.items():
    print(f"{name:<10} {profile.vpip:>5.0%} {profile.pfr:>5.0%} {profile.aggression:>6.1f} {profile.fold_to_cbet:>7.0%} {profile.three_bet_pct:>5.0%}")

## 4. Task Generation Demo

In [ ]:
# Generate a task with full reasoning trace
context, question, answer, trace = generate_poker_task_with_trace(
    street='flop', num_history_hands=15
)

print("=== CONTEXT (game state + hand history) ===")
print(context)
print(f"\n=== QUESTION ===")
print(question)
print(f"\n=== HEURISTIC ANSWER: {answer} ===")
print(f"\n=== REASONING TRACE ===")
print(trace)

## 5. Hand Evaluation Demo

In [ ]:
# Demonstrate hand evaluation
test_hands = [
    ([Card.from_str('Ah'), Card.from_str('Kh')], [Card.from_str('Kd'), Card.from_str('7c'), Card.from_str('2s')], "Top pair top kicker"),
    ([Card.from_str('9h'), Card.from_str('9d')], [Card.from_str('9c'), Card.from_str('5h'), Card.from_str('2d')], "Set of nines"),
    ([Card.from_str('7h'), Card.from_str('2d')], [Card.from_str('Kd'), Card.from_str('Qs'), Card.from_str('Jc')], "Air (nothing)"),
]

print(f"{'Description':<25} {'Hand':>12} {'Board':>15} {'Ranking':<20} {'Equity':>8}")
print("-" * 85)
for hole, board, desc in test_hands:
    score = HandEvaluator.evaluate(hole + board)
    name = HandEvaluator.hand_name(score)
    equity = HandEvaluator.equity_estimate(hole, board, num_simulations=500)
    hole_str = ' '.join(str(c) for c in hole)
    board_str = ' '.join(str(c) for c in board)
    print(f"{desc:<25} {hole_str:>12} {board_str:>15} {name:<20} {equity:>7.1%}")

## 6. Heuristic Agent Evaluation

In [ ]:
agent = PokerHeuristicAgent()
ev = PokerEvaluationFramework(
    agents=[agent],
    task_generator=generate_poker_task,
    num_episodes=50,
)
ev.run_evaluation()
ev.display_results()
ev.display_confusion_matrix('PokerHeuristicAgent')

## 7. Agent REPL Transcript

The heuristic agent generates executable Python code for each step.
This is what behavior cloning will teach the LLM to produce.

In [ ]:
# Show one full episode transcript
context, question, answer = generate_poker_task(street='flop')
predicted, transcript = agent.run_episode(context, question, answer)

print(f"Correct: {answer}, Predicted: {predicted}")
print()

for entry in transcript:
    print(f"{'='*60}")
    print(f"STEP {entry['step']}: {entry['action']}")
    print(f"{'='*60}")
    print(f"Code:\n{entry['code'][:400]}...")
    er = entry['exec_result']
    print(f"\nExecution: ok={er['ok']}, time={er['runtime_sec']:.4f}s")
    if er['stdout']:
        print(f"stdout:\n{er['stdout'].strip()[:300]}")
    if er['stderr']:
        print(f"stderr: {er['stderr'].strip()[:200]}")
    print()

## 8. Trajectory Collection

In [ ]:
# Collect BC training data
trajectories = collect_poker_trajectories_with_traces(num_episodes=100)

correct = sum(1 for t in trajectories if t.is_correct)
parsed = sum(1 for t in trajectories if t.parsed_stats)

print(f"Trajectories: {len(trajectories)}")
print(f"Correct: {correct} ({correct/len(trajectories):.0%})")
print(f"Parsed opponent stats: {parsed} ({parsed/len(trajectories):.0%})")

# Action distribution
from collections import Counter
action_dist = Counter(t.action_type for t in trajectories)
print(f"\nAction distribution:")
for action, count in action_dist.most_common():
    print(f"  {action}: {count} ({count/len(trajectories):.0%})")

## 9. Reward Function Demo

In [ ]:
# Show reward computation for different scenarios
scenarios = [
    ("raise $10", "raise $10", True,  "Exact match, parsed stats"),
    ("call $10",  "raise $10", True,  "Call vs raise (both staying in), parsed"),
    ("fold",      "raise $10", False, "Fold vs raise, no stats"),
    ("raise $10", "fold",      False, "Raise vs fold, no stats"),
    ("fold",      "fold",      False, "Correct fold, no stats"),
]

print(f"{'Predicted':<12} {'Correct':<12} {'Stats':>6} {'Reward':>8} {'Note'}")
print("-" * 65)
for pred, corr, stats, note in scenarios:
    r = 0.5 * (1.0 if parse_action(pred)[0] == parse_action(corr)[0] else 0.0) + 0.2 * (0.2 if stats else 0.0)
    r_simple = compute_poker_reward_simple(pred, corr)
    print(f"{pred:<12} {corr:<12} {str(stats):>6} {r_simple:>8.2f} {note}")

## 10. LLM Agent (Zero-Shot Baseline)

Run the LLM agent on poker tasks to establish a pre-training baseline.
Requires `HF_TOKEN` environment variable.

In [ ]:
if HF_TOKEN_SET:
    llm_agent = PokerLLMAgent(max_steps=5)
    heuristic_agent = PokerHeuristicAgent()
    
    ev_compare = PokerEvaluationFramework(
        agents=[heuristic_agent, llm_agent],
        task_generator=generate_poker_task,
        num_episodes=20,
    )
    ev_compare.run_evaluation()
    ev_compare.display_results()
    
    print("\n--- LLM Confusion Matrix ---")
    ev_compare.display_confusion_matrix('PokerLLMAgent')
else:
    print("Skipped — set HF_TOKEN to run LLM evaluation")
    print("  export HF_TOKEN=your_token_here")

## 11. Behavior Cloning (GPU Required)

Fine-tune Qwen 1.5B on heuristic trajectories. Requires GPU (Colab).

In [ ]:
if GPU_AVAILABLE:
    from src.training import load_model
    
    # Load model
    model, tokenizer = load_model(
        model_id_or_path="Qwen/Qwen2.5-Coder-1.5B-Instruct",
        load_in_4bit=True,
    )
    
    # Collect training data
    print("Collecting heuristic trajectories...")
    bc_trajectories = collect_poker_trajectories_with_traces(num_episodes=500)
    print(f"Collected {len(bc_trajectories)} trajectories")
    
    # Train
    bc_trainer = PokerBCTrainer(
        model=model,
        tokenizer=tokenizer,
        output_dir="./checkpoints/poker_bc",
        num_epochs=3,
        batch_size=4,
    )
    bc_result = bc_trainer.train(bc_trajectories)
    print(f"\nBC Training complete: loss={bc_result['train_loss']:.4f}")
else:
    print("Skipped — GPU required for training")
    print("Run this notebook on Google Colab with a GPU runtime")

## 12. REINFORCE (GPU Required)

After BC warm-start, run REINFORCE to optimize beyond the heuristic.

In [ ]:
if GPU_AVAILABLE:
    rl_trainer = PokerReinforceTrainer(
        model=model,
        tokenizer=tokenizer,
        output_dir="./checkpoints/poker_rl",
        batch_size=8,
        learning_rate=1e-5,
    )
    rl_history = rl_trainer.train(num_iterations=20)
    
    # Plot training curves
    try:
        import matplotlib.pyplot as plt
        
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        iters = range(1, len(rl_history) + 1)
        
        axes[0].plot(iters, [h['accuracy'] for h in rl_history], 'b-o')
        axes[0].set_title('Action Accuracy')
        axes[0].set_xlabel('Iteration')
        axes[0].set_ylim(0, 1.1)
        
        axes[1].plot(iters, [h['avg_reward'] for h in rl_history], 'g-o')
        axes[1].set_title('Average Reward')
        axes[1].set_xlabel('Iteration')
        
        axes[2].plot(iters, [h['loss'] for h in rl_history], 'r-o')
        axes[2].set_title('Policy Loss')
        axes[2].set_xlabel('Iteration')
        
        plt.tight_layout()
        plt.savefig('../figures/poker_rl_training.png', dpi=100, bbox_inches='tight')
        plt.show()
    except ImportError:
        for i, h in enumerate(rl_history):
            print(f"Iter {i+1}: acc={h['accuracy']:.1%} reward={h['avg_reward']:.3f}")
else:
    print("Skipped — GPU required")

## 13. Opponent Adjustment Examples

Show cases where the heuristic changes its decision based on opponent history.

In [ ]:
# Find and display adjustment examples
found = 0
for seed in range(500, 2000):
    random.seed(seed)
    try:
        _, _, answer, trace = generate_poker_task_with_trace(num_history_hands=20)
    except Exception:
        continue
    if 'no adjustment' not in trace and 'no exploitable' not in trace and 'insufficient' not in trace:
        found += 1
        print(f"--- Example {found} (seed {seed}) ---")
        print(f"Answer: {answer}")
        print(trace)
        print()
        if found >= 4:
            break

## 14. Current Limitations and Next Steps

### What Works
- Poker environment generates realistic scenarios with structured hand histories
- Heuristic bot follows retrieve → compute → decide with opponent modeling
- Training pipeline (BC + REINFORCE) is wired up and ready for GPU
- Agent generates executable REPL code that parses context

### Current Limitations
- Opponent adjustments fire ~9% of the time — may need data augmentation
- No LLM zero-shot numbers yet (pending API evaluation)
- EV-based RL reward uses equity proxy, not full hand simulation
- Hand equity estimation is slow (Monte Carlo) — may bottleneck RL training

### Next Steps
1. **Week 11**: Run zero-shot LLM evaluation, begin BC on Colab
2. **Week 12**: Evaluate BC model, start REINFORCE
3. **Weeks 13-14**: Full comparison: heuristic vs zero-shot vs BC vs RL